# YEJI v9 AWQ 양자화

- **입력**: `tellang/yeji-4b-instruct-v9` (fp16)
- **출력**: `tellang/yeji-4b-instruct-v9-AWQ` (4-bit AWQ)
- **런타임**: GPU (A100 권장, T4도 가능)
- **예상 시간**: A100 ~5분, T4 ~15분

## 1. 설치

In [ ]:
!pip install autoawq transformers accelerate huggingface_hub -q

## 2. GPU 확인

In [ ]:
!nvidia-smi
import torch
print(f"\nCUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 3. HuggingFace 로그인

In [ ]:
from huggingface_hub import login
login()

## 4. 모델 로딩

In [ ]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

MODEL_ID = "tellang/yeji-4b-instruct-v9"

print(f"모델 로딩: {MODEL_ID}")
model = AutoAWQForCausalLM.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
print("모델 로딩 완료!")

## 5. AWQ 양자화

In [ ]:
quant_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM",
}

print("AWQ 양자화 시작...")
model.quantize(tokenizer, quant_config=quant_config)
print("양자화 완료!")

## 6. 로컬 저장

In [ ]:
OUTPUT_DIR = "yeji-4b-instruct-v9-AWQ"

print(f"저장 중: {OUTPUT_DIR}")
model.save_quantized(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("저장 완료!")

# 파일 확인
import os
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024**2
    print(f"  {f}: {size:.1f} MB")

## 7. HuggingFace 업로드

In [ ]:
from huggingface_hub import HfApi

HF_REPO = "tellang/yeji-4b-instruct-v9-AWQ"

api = HfApi()
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=HF_REPO,
    commit_message="feat: AWQ 4-bit 양자화 (w4, g128, GEMM)",
)
print(f"업로드 완료! https://huggingface.co/{HF_REPO}")

## 8. (선택) 빠른 테스트

In [ ]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

# 양자화된 모델 로딩
model = AutoAWQForCausalLM.from_quantized(OUTPUT_DIR, fuse_layers=False)
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=True)

# 테스트 프롬프트
prompt = "1990년 3월 15일생 남성의 오늘 운세를 알려주세요."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))